In [1]:
import numpy as np
import pandas as pd

# Step 1. Calcualte EMA Crossover
### $EMA(P_t\frac{1}{n_{k,s}}) - EMA(P_t\frac{1}{n_{k,l}})$
- x > 0: long
- x < 0: short

In [25]:
EMA_PAIRS = [(8, 24), (16, 48), (32, 96)] # (n_ks, n_kl)

def ema(x, n):
    return x.ewm(alpha=1/n, adjust=False).mean() # Exponentially Weighted Moving

def ema_crossovers(price):
    signals = {}
    
    for k, (n_s, n_l) in enumerate(EMA_PAIRS, start=1):
        
        short = ema(price, n_s)
        long = ema(price, n_l)
        
        signals[f"x{k}"] = short - long
        
    return signals

# Step 2. First Volatility Normalization
### $y_{k,t} = \frac{x_{k,t}}{\sigma_{63}(P_t)}$
- 63 days for conventional markets: 3 months of market arctivity
- 91 for crypto market: market never closes


In [22]:
def first_norm(price, x, crypto=True):
    
    window = 91 if crypto else 63
    
    price_vol = price.rolling(window).std()
    
    return x /price_vol

# Step 3. Normalize Entire Signal
### $z_{k,t} = \frac{y_{k,t}}{\sigma(y_k)}$ 
- 252 days for conventional markets
- 365 days for crypto market

In [4]:
def second_norm(y, crypto=True):
    
    window = 365 if crypto else 252
    
    rolling_vol = y.rolling(window).std()
    
    return y / rolling_vol

# Step 4. Nonlinear Response Function
###  $u(z) =\frac{ze^{-\frac{z^2_k}{4} }}{\sqrt{2}e^{-\frac{1}{2}}}~~$    $~-1\le u \le 1$

In [5]:
def u_func(z):
    den = np.sqrt(2) * np.exp(-0.5)
    
    return z * np.exp(-(z ** 2) / 4) / den

# Step 5. Create Combined Signal
### $\text{Signal}_t = \frac{1}{3}u_{1,t} + \frac{1}{3}u_{2,t} + \frac{1}{3}$

In [23]:
def momentum_signal(price, crytpo=True):
    
    xs = ema_crossovers(price)
    
    us = []
    
    for x in xs.values():
        
        y = first_norm(price, x, crypto=crytpo)
        
        z = second_norm(y, crypto=crytpo)
        
        u = u_func(z)
        
        us.append(u)
        
    return sum(us) / len(us)

# Portfolio Construction

## Strategy Returns with Signal Lag
### $\text{Signal}_{t-1}R_t$<br>

## Time-series Portfolio
### $w_{i,t}=\frac{\text{Signal}_{i,t}}{N}$

In [8]:
def ts_portfolio(signals, returns):
    
    n = signals.notna().sum(axis=1)
    
    weights = signals.div(n, axis=0)
    
    weights = weights.fillna(0)
    
    port_returns = (weights.shift(1) * returns).sum(axis=1, min_count=1)
    
    return port_returns, weights

## Cross-sectional Portfolio

In [9]:
def cs_weights(signals, n_l=3, n_s=3):
    
    weights = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)
    
    total_positions = n_l + n_s
    position_size = 1 / total_positions
    
    for date, row in signals.iterrows():
        
        valid = row.dropna()
        if len(valid) < total_positions:
            continue
        
        l_assets = valid.nlargest(n_l).index
        s_assets = valid.nsmallest(n_s).index
        
        weights.loc[date, l_assets] = position_size
        weights.loc[date, s_assets] = -position_size
        
    return weights

def cs_portfolio(signals, returns):
    
    weights = cs_weights(signals)
    
    port_returns = (weights.shift(1) * returns).sum(axis=1, min_count=1)
    
    return port_returns, weights
    

# Rets and Metrics
## Data

In [10]:
from binance.client import Client as bnb_client
import os

### API Object

In [11]:
API_KEY = os.getenv('BINANCE_API_KEY')
API_SECRET = os.getenv('BINANCE_API_SECRET')

client = bnb_client(API_KEY, API_SECRET, tld='us')

In [12]:
univ = [ 'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'BCHUSDT', 'AAVEUSDT', 'LTCUSDT', 
        'AVAXUSDT', 'LINKUSDT', 'ETCUSDT', 'XRPUSDT', 'NEOUSDT', 'XLMUSDT', 'ADAUSDT',
        'ATOMUSDT', 'NEARUSDT', 'FETUSDT', 'SUIUSDT', 'DASHUSDT', 'GRTUSDT', 'ICPUSDT',
        'DOTUSDT', 'UNIUSDT', 'VETUSDT', 'CHZUSDT', 'FILUSDT', 'THETAUSDT', 'DIAUSDT',
        'APTUSDT', 'ZENUSDT']

In [33]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def get_binance_px(symbol:str, freq:str, start_ts:str,end_ts:str) -> pd.DataFrame:
    
    data = client.get_historical_klines(symbol, freq, start_ts, end_ts)
    columns = ['open_time','open','high','low','close','volume','close_time','quote_volume',
    'num_trades','taker_base_volume','taker_quote_volume','ignore']

    data = pd.DataFrame(data, columns = columns)
    
    # Convert from POSIX timestamp (number of millisecond since jan 1, 1970)
    data['open_time'] = pd.to_datetime(data['open_time'], unit='ms')
    data['close_time'] = pd.to_datetime(data['close_time'], unit='ms')
    
    # enforce data types
    float_cols = ['open','high','low','close','volume',
                    'quote_volume','taker_base_volume','taker_quote_volume']
    
    data[float_cols] = data[float_cols].astype(float)
    
    return data

end = pd.to_datetime('2024-05-01')
start = end - pd.DateOffset(years=2)
start, end = start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')

freq = '1d'

def fetch_symbol(symbol):
    for attempt in range(3):
        try:
            data = get_binance_px(symbol, freq, start_ts=start, end_ts=end)
            return symbol, data.set_index('open_time')['close']
        except Exception as _:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                raise

px = {}
with ThreadPoolExecutor(max_workers=5) as executor: # network threading
    futures = {executor.submit(fetch_symbol, x): x for x in univ}
    for future in as_completed(futures):
        symbol, series = future.result()
        px[symbol] = series
    
px = pd.DataFrame(px).sort_index()
px.to_pickle('training_data.pk')


In [13]:
px = pd.read_pickle('training_data.pk')

## Rets

In [34]:

returns = px.pct_change(fill_method=None)
print("Num of observations: ", returns.shape[0])
print("Num of assets: ", returns.shape[1])
print(f"Time period start: {returns.index[0]}      Time period end: {returns.index[-1]}")
returns.head()

Num of observations:  732
Num of assets:  30
Time period start: 2022-05-01 00:00:00      Time period end: 2024-05-01 00:00:00


,BNBUSDT,BTCUSDT,ETHUSDT,BCHUSDT,SOLUSDT,AAVEUSDT,AVAXUSDT,LTCUSDT,ETCUSDT,LINKUSDT,...,ICPUSDT,UNIUSDT,DOTUSDT,VETUSDT,CHZUSDT,FILUSDT,THETAUSDT,DIAUSDT,APTUSDT,ZENUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2022-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-02,-0.000792,0.000829,0.011203,0.003664,-0.024358,-0.016248,0.027344,0.011881,-0.004888,-0.014172,...,NaN,-0.027463,-0.024104,-0.019339,-0.019759,-0.021941,NaN,NaN,NaN,-0.021793
2022-05-03,-0.014352,-0.020052,-0.026262,-0.020778,-0.019328,-0.015402,-0.013887,-0.010448,-0.028944,-0.001797,...,NaN,-0.007909,-0.016021,-0.014727,-0.003359,-0.013596,NaN,NaN,NaN,0.000595
2022-05-04,0.049533,0.051020,0.057475,0.076669,0.080773,0.130309,0.126907,0.068577,0.151019,0.097210,...,NaN,0.158860,0.105834,0.153387,0.101124,0.082012,NaN,NaN,NaN,0.113178
2022-05-05,-0.060372,-0.078475,-0.066070,-0.080698,-0.088462,-0.116851,-0.122136,-0.089301,-0.077831,-0.105824,...,NaN,-0.081772,-0.104294,-0.118675,-0.100000,-0.112102,NaN,NaN,NaN,-0.078494


In [ ]:
signals = momentum_signal(px, crytpo=True)

ts_rets, ts_weights = ts_portfolio(signals, returns)

cs_rets, cs_wgts = cs_portfolio(signals, returns)

## Metrics


In [17]:
def metrics(rets, crypto=True):
    
    periods = 365 if crypto else 252
    
    ann_rets = rets.mean() * periods
    ann_vol = rets.std() * np.sqrt(periods)
    sharpe = ann_rets / ann_vol
    
    return {"Annual Return": ann_rets,
            "Annual Volatility": ann_vol,
            "Sharpe": sharpe}

### Before Tcosts 

In [19]:
from pprint import pprint as pp

In [37]:
pp(metrics(ts_rets))

{'Annual Return': 0.2368477941703544,
 'Annual Volatility': 0.26713756940255357,
 'Sharpe': 0.886613570304089}


In [38]:
pp(metrics(cs_rets))

{'Annual Return': -0.07383435493638617,
 'Annual Volatility': 0.16294535748589986,
 'Sharpe': -0.45312340330270084}


## Backtest

In [53]:
def backtest(wgts, rets, bps=20):
    
    returns = rets.reindex(index=wgts.index, columns=wgts.columns)
    
    held_weights = wgts.shift(1)
    
    gross_returns = (held_weights * returns).sum(axis=1, min_count=1)
    
    turnover = wgts.diff().abs().sum(axis=1)
    
    tcosts = turnover * (bps / 10_000)
    
    tcosts = tcosts.shift(1).fillna(0)
    
    net_returns = gross_returns - tcosts
    
    return net_returns

### Validation Data

In [66]:
end = pd.to_datetime('2026-08-01')
start = end - pd.DateOffset(years=2)
start, end = start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')
val_px = {}
with ThreadPoolExecutor(max_workers=5) as executor: # network threading
    futures = {executor.submit(fetch_symbol, x): x for x in univ}
    for future in as_completed(futures):
        symbol, series = future.result()
        val_px[symbol] = series
    
val_px = pd.DataFrame(val_px).sort_index()
val_px.to_pickle('validation_data.pk')

In [67]:
val_px = pd.read_pickle("validation_data.pk")

In [68]:
val_returns = val_px.pct_change(fill_method=None)
print("Num of observations: ", val_returns.shape[0])
print("Num of assets: ", val_returns.shape[1])
print(f"Time period start: {val_returns.index[0]}      Time period end: {val_returns.index[-1]}")
val_returns.head()

Num of observations:  731
Num of assets:  30
Time period start: 2024-08-01 00:00:00      Time period end: 2026-08-01 00:00:00


,BTCUSDT,SOLUSDT,BCHUSDT,ETHUSDT,BNBUSDT,ETCUSDT,AAVEUSDT,AVAXUSDT,LINKUSDT,LTCUSDT,...,ICPUSDT,CHZUSDT,UNIUSDT,DOTUSDT,VETUSDT,FILUSDT,ZENUSDT,DIAUSDT,APTUSDT,THETAUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08-02,-0.058980,-0.084202,-0.078133,-0.067487,-0.055081,-0.070866,-0.042676,-0.086281,-0.077239,-0.065383,...,0.003026,-0.046589,-0.073597,-0.041932,-0.074060,-0.061410,0.003109,0.000000,-0.056250,-0.098529
2024-08-03,-0.013742,-0.068206,-0.046445,-0.027348,-0.024534,-0.026919,-0.027039,-0.028499,-0.016959,-0.006150,...,-0.131061,-0.038394,0.005135,-0.014654,-0.060705,-0.043444,-0.133264,-0.053408,-0.091060,-0.022838
2024-08-04,-0.041753,-0.029756,-0.096588,-0.072388,-0.067511,-0.025102,-0.087128,-0.060858,-0.070373,-0.034035,...,-0.058249,0.009074,-0.075733,-0.063256,-0.033188,-0.033105,0.000000,-0.064526,-0.025501,-0.044240
2024-08-05,-0.072432,-0.061338,-0.056655,-0.106427,-0.058203,-0.081450,-0.010079,-0.089510,-0.129995,-0.100577,...,-0.009011,-0.086331,-0.127947,-0.094200,-0.024390,-0.091681,-0.213349,-0.066978,-0.061682,-0.056769


In [69]:
val_signals = momentum_signal(val_px, crytpo=True)

val_ts_rets, val_ts_weights = ts_portfolio(val_signals, returns)

val_cs_rets, val_cs_wgts = cs_portfolio(val_signals, returns)

In [70]:
train_ts_backtest = backtest(ts_weights, returns)
train_cs_backtest = backtest(cs_wgts, returns)

val_ts_backtest = backtest(val_ts_weights, val_returns)
val_cs_backtest = backtest(val_cs_wgts, val_returns)

### Metrics and $\alpha$

In [83]:
print("---Training Metrics---")
print('\n--Time-series Portfolio--')
pp(metrics(train_ts_backtest))
print('\n--Cross-sectional Portfolio--')
pp(metrics(train_cs_backtest))

---Training Metrics---

--Time-series Portfolio--
{'Annual Return': 0.23004869348987372,
 'Annual Volatility': 0.2671187573753156,
 'Sharpe': 0.8612225354382114}

--Cross-sectional Portfolio--
{'Annual Return': -0.11377963537414268,
 'Annual Volatility': 0.16318040099440098,
 'Sharpe': -0.6972628739774127}


In [84]:
print("---Validation Metrics---")
print('\n--Time-series Portfolio--')
pp(metrics(val_ts_backtest))
print('\n--Cross-sectional Portfolio--')
pp(metrics(val_cs_backtest))

---Validation Metrics---

--Time-series Portfolio--
{'Annual Return': 0.07748943856932425,
 'Annual Volatility': 0.25579731878204387,
 'Sharpe': 0.30293295855595087}

--Cross-sectional Portfolio--
{'Annual Return': 0.003075535305798419,
 'Annual Volatility': 0.2543475872623964,
 'Sharpe': 0.012091859564704887}


In [76]:
import statsmodels.api as sm

def alpha(bench, strat_ret, crypto=True):
    benchmark = bench["BTCUSDT"].copy()
    strat_ret.name = 'strat'
    benchmark.name = 'bench'
    
    df = pd.concat([benchmark, strat_ret], axis=1, join="inner")
    
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    
    X = sm.add_constant(df['bench'])
    model = sm.OLS(df['strat'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    
    daily_alpha = model.params['const']
    beta = model.params['bench']
    
    periods = 365 if crypto else 252
    
    annual_alpha = daily_alpha * periods
    
    print(f'Alpha (ann.): {annual_alpha:.4f}')
    print(f'Alpha t-stat: {model.tvalues['const']:.3f}')
    print(f'Beta vs bench: {beta:.4f}')
    print(f'R2: {model.rsquared:.4f}')
    print(f'Correlation: {df['strat'].corr(df['bench']):.4f}')
    print()

In [81]:
print("---Training Alpha---")
print('\n--Time-series Portfolio--')
alpha(returns, train_ts_backtest)
print('\n--Cross-sectional Portfolio--')
alpha(returns, train_cs_backtest)

---Training Alpha---

--Time-series Portfolio--
Alpha (ann.): 0.1741
Alpha t-stat: 0.969
Beta vs bench: 0.1576
R2: 0.1018
Correlation: 0.3191


--Cross-sectional Portfolio--
Alpha (ann.): -0.1306
Alpha t-stat: -1.138
Beta vs bench: 0.0473
R2: 0.0246
Correlation: 0.1569



In [82]:
print("---Validation Alpha---")
print('\n--Time-series Portfolio--')
alpha(val_returns, val_ts_backtest)
print('\n--Cross-sectional Portfolio--')
alpha(val_returns, val_cs_backtest)

---Validation Alpha---

--Time-series Portfolio--
Alpha (ann.): 0.0996
Alpha t-stat: 0.687
Beta vs bench: -0.2631
R2: 0.2211
Correlation: -0.4702


--Cross-sectional Portfolio--
Alpha (ann.): 0.0066
Alpha t-stat: 0.031
Beta vs bench: -0.0424
R2: 0.0058
Correlation: -0.0762

